# Me-Mex inference stack (Colab L4)

Runs **four capabilities** behind one **cloudflared** URL:

| # | Service | Colab process | Local consumer |
|---|---------|---------------|----------------|
| 1 | Chat LLM | vLLM `:8000` (GPU) | `llm_gateway` → `COLAB_VLLM_URL/v1/chat/completions` |
| 2 | Decision reranker | gateway CPU (`reranker_service`) | `HttpDecisionScorer` → `…/v1/decision/score` |
| 3 | STT | `speech_sidecar` (CPU) | Phase 2 worker → `…/stt/transcribe` |
| 4 | TTS | `speech_sidecar` (CPU) | Phase 2 worker → `…/tts/synthesize` |

Read **[TOPOLOGY.md](./TOPOLOGY.md)** for architecture, LiveKit vs HTTP, and local `.env`.

**Order:** §1 GPU → §2 deps → §5 vLLM → §6 speech → §7 gateway (BGE reranker) → §8 tunnel → §9 smoke

Operator guide: **[RUNBOOK.md](./RUNBOOK.md)** (tmux + `colab/logs/` for multi-process debug)

## §0 — Clone repo (or upload `colab/` files)

In [ ]:
# Option A: git clone (set your remote)
# !git clone https://github.com/YOUR_USER/me-mex.git /content/me-mex

# Option B: upload me-mex/colab/{inference_gateway.py,speech_sidecar.py} to /content/colab
import os
COLAB_DIR = "/content/me-mex/colab" if os.path.isdir("/content/me-mex/colab") else "/content/colab"
os.chdir(COLAB_DIR)
print("Working directory:", os.getcwd(), os.listdir("."))

## §1 — GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 or L4 GPU"

## §2 — Dependencies

Pins: [requirements-colab.txt](./requirements-colab.txt) (core) + [requirements-colab-speech.txt](./requirements-colab-speech.txt) (Phase 2). **Install spacy wheels before kokoro** — see [RUNBOOK §2.1](./RUNBOOK.md).

In [ ]:
!pip install -q -r requirements-colab.txt
!pip uninstall -y torchaudio 2>/dev/null || true

In [ ]:
!pip install -q "spacy==3.8.18" "thinc>=8.3.12,<8.4" --only-binary=:all:
!pip install -q -r requirements-colab-speech.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # gates for Ministral weights

## §3 — (Notebook continues at §5)

Decision routing is the BGE cross-encoder in the gateway (§7). Optional held-out eval runs on your Mac — **[RUNBOOK.md](./RUNBOOK.md) §6**.

## §5 — vLLM (GPU): chat

**VRAM tips (L4 24GB):** use Instruct + tool parser; if OOM, add `--quantization fp8`, lower `--max-model-len` / `--gpu-memory-utilization`.

Runs in a **background shell** (blocking cell).

In [ ]:
VLLM_MODEL = "mistralai/Ministral-3-8B-Instruct-2512"

vllm_cmd = f"""
vllm serve {VLLM_MODEL} \
    --host 127.0.0.1 --port 8000 \
    --tokenizer-mode mistral \
    --config-format mistral \
    --load-format mistral \
    --enable-auto-tool-choice \
    --tool-call-parser mistral \
    --max-model-len 8192 \
    --max-num-seqs 4 \
    --gpu-memory-utilization 0.90 \
    --enable-prefix-caching
"""
print(vllm_cmd)

In [ ]:
get_ipython().system_raw(vllm_cmd + " &")
import time, httpx
for _ in range(120):
    try:
        r = httpx.get("http://127.0.0.1:8000/v1/models", timeout=2)
        if r.status_code == 200:
            print("vLLM ready", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("vLLM did not become ready in time")

## §6 — Speech sidecar (CPU): Whisper-tiny + Kokoro

Does **not** use GPU. First request downloads small models.

In [ ]:
import subprocess, time, httpx
speech = subprocess.Popen(
    ["uvicorn", "speech_sidecar:app", "--host", "127.0.0.1", "--port", "7861"],
    cwd=COLAB_DIR,
)
for _ in range(60):
    try:
        r = httpx.get("http://127.0.0.1:7861/health", timeout=2)
        if r.status_code == 200:
            print("speech_sidecar", r.json())
            break
    except Exception:
        pass
    time.sleep(1)

## §7 — Inference gateway (public surface)

Cloudflared should point here — **not** directly to vLLM — so `/v1/decision/score` (BGE cross-encoder on CPU) and speech paths share one hostname. First `/health` may take a minute while the reranker downloads.

In [ ]:
import os, subprocess, time, httpx
env = os.environ.copy()
env.update({
    "VLLM_BASE": "http://127.0.0.1:8000/v1",
    "SPEECH_BASE": "http://127.0.0.1:7861",
    "LLM_MODEL_NAME": VLLM_MODEL,
    "BGE_RERANKER_MODEL": "BAAI/bge-reranker-base",
})
gw = subprocess.Popen(
    ["uvicorn", "inference_gateway:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd=COLAB_DIR,
    env=env,
)
time.sleep(2)
print(httpx.get("http://127.0.0.1:7860/health").json())

## §8 — cloudflared quick tunnel

Same pattern as your chat-only setup; target port **7860** (gateway).

Copy the `https://….trycloudflare.com` URL into local `backend/.env`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
# Run in background; read URL from cell output
get_ipython().system_raw("cloudflared tunnel --url http://127.0.0.1:7860 &")

### Local `.env` (paste your tunnel host)

```bash
COLAB_VLLM_URL=https://YOUR_SUBDOMAIN.trycloudflare.com/v1
CHAT_LLM_MODE=live
DECISION_SCORER_MODE=live
LLM_MODEL_NAME=mistralai/Ministral-3-8B-Instruct-2512

# Phase 2 LiveKit worker (same hostname, no /v1):
# COLAB_SPEECH_BASE_URL=https://YOUR_SUBDOMAIN.trycloudflare.com
```

## §9 — Smoke tests (Colab or local)

Replace `PUBLIC_BASE` with your trycloudflare host **without** trailing slash.

In [ ]:
PUBLIC_BASE = "https://YOUR_SUBDOMAIN.trycloudflare.com"  # set me
import httpx
h = {"Bypass-Tunnel-Reminder": "true", "User-Agent": "Me-Mex-Client"}
models = httpx.get(f"{PUBLIC_BASE}/v1/models", headers=h, timeout=30)
print("models", models.status_code)
dec = httpx.post(
    f"{PUBLIC_BASE}/v1/decision/score",
    headers=h,
    json={
        "query_node": {"id": "q", "title": "query", "node_kind": "ROOT"},
        "frontier": [
            {"id": "a", "title": "A", "node_kind": "CONCEPT_M"},
            {"id": "b", "title": "B", "node_kind": "CONCEPT_M"},
        ],
    },
    timeout=120,
)
print("decision", dec.status_code, dec.json())

In [ ]:
# Local Phase 1 exit gate (on your Mac, not Colab)
# cd me-mex/backend && DECISION_SCORER_MODE=live uv run pytest tests/integration/decision -m vllm

## §10 — Voice path (Phase 2) on local worker

LiveKit **WebRTC stays on your Mac**. The worker only adds HTTP calls to Colab:

```text
mic → LiveKit → worker → POST {COLAB_SPEECH_BASE_URL}/stt/transcribe → text
     → LangGraph + COLAB_VLLM_URL (chat / tools)
     → POST {COLAB_SPEECH_BASE_URL}/tts/synthesize → audio → LiveKit publish
```

No extra tunnel type is required. Use **short utterances** or sentence-chunked TTS to limit RTT. For lowest latency, run STT/TTS **locally** ([M1](../docs/10-open-questions.md)) and keep only vLLM on Colab.

Barge-in: cancel in-flight TTS HTTP and LangGraph foreground stream locally — Colab is stateless.

---

**Related:** [local_llm.ipynb](./local_llm.ipynb) (minimal Reasoning-only vLLM sample), [RUNBOOK.md](./RUNBOOK.md), [TOPOLOGY.md](./TOPOLOGY.md).